In [ ]:
import torch
from torch.utils.data import DataLoader
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

import torchvision.datasets as datasets
from torchvision.transforms import ToTensor

In [ ]:
# In case you want to do the computations on Cuda/mps instead of CPU

device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.mps.is_available():
    device = torch.device("mps")
print("Running on device:", device)

In [ ]:
# Downloading the FashionMNIST dataset and also transforming it to tensor

mnist_train = datasets.FashionMNIST(
    root="./data", download=True, train=True, transform=ToTensor()
)
mnist_test = datasets.FashionMNIST(
    root="./data", download=True, train=False, transform=ToTensor()
)

# packing the examples in each dataset to batch sizes of 32
train_dataloader = DataLoader(mnist_train, batch_size=32, shuffle=True)
test_dataloader = DataLoader(mnist_test, batch_size=32, shuffle=True)

In [ ]:
model = nn.Sequential(
    nn.Sequential(
        # the CNN layer for pattern detection in images using filters
        nn.Conv2d(1, 32, kernel_size=(3, 3), padding=1, padding_mode="reflect"),
        # Max pooling layer to reduce the shape of the output matrice of the CNN layer
        nn.MaxPool2d(kernel_size=2),
        # Normalization layer to normalize the values of the output matrice
        nn.BatchNorm2d(32),
        # to add non-linearity
        nn.ReLU(),
        # to disable randomly a few neuron connections in the training process to improve generalization
        nn.Dropout(0.1),
    ),
    nn.Sequential(
        nn.Conv2d(32, 64, kernel_size=(3, 3), padding=1, padding_mode="reflect"),
        nn.MaxPool2d(kernel_size=2),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.Dropout(0.1),
    ),
    # Now flattening the output matrice values in order to feed it to Dense neural netwok as a vector and not a 3D matrice
    nn.Flatten(),
    nn.Sequential(
        nn.Linear(64 * 7 * 7, 1000),
        nn.BatchNorm1d(1000),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(1000, 100),
        nn.BatchNorm1d(100),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(100, 10),
    ),
).to(device)
print(model)

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for i in range(0, 20):
    model.train()

    loss_sum = 0
    for X, Y in train_dataloader:
        # using one hot encoding to convert output category values into vectors like: (0,0,1,0,0,0,0,0,0,0)
        Y = F.one_hot(y, num_classes=10).type(torch.float32).to(device)
        X = X.to(device)

        optimizer.zero_grad()
        outputs = model(X)
        loss = loss_fn(outputs, Y)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item()
    print(loss_sum)

In [ ]:
# Evaluating the model on validation dataset
model.eval()
with torch.no_grad():
    accurate = 0
    total = 0
    for X, Y in test_dataloader:
        X = X.to(device)
        Y = Y.to(device)
        outputs = nn.functional.softmax(model(X), dim=1)
        correct_pred = Y == outputs.max(dim=1).indices
        total += correct_pred.size(0)
        accurate += correct_pred.type(torch.int).sum().item()
    print("Accuracy on validation data:", accurate / total)